# AGGREGATIONS
    - Spark also allows us to create the following groupings types:
        a.The simplest grouping is to just summarize a complete DataFrame by performing an aggregation in a select statement.
        b.A “group by” allows you to specify one or more keys as well as one or more aggregation functions to transform the value columns.
        c.A “window” gives you the ability to specify one or more keys as well as one or more aggregation functions to transform the value columns. However, the rows input to the function are somehow related to the current row.

In [1]:
from pyspark.sql import SparkSession
spark=SparkSession.builder.appName('Agg').getOrCreate();

26/04/01 15:07:42 WARN Utils: Your hostname, intern7 resolves to a loopback address: 127.0.0.1; using 10.10.42.90 instead (on interface enp2s0)
26/04/01 15:07:42 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/01 15:07:43 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
df=spark.read.format('csv').load('file:///home/unik/PYSPARK/all/*csv',header=True,inferSchema=True).coalesce(5);
df.cache();
df.createOrReplaceTempView('dfTable')

In [3]:
df.show()

[Stage 2:===============================================>           (4 + 1) / 5]

+---------+---------+--------------------+--------+--------------+---------+----------+--------------+
|InvoiceNo|StockCode|         Description|Quantity|   InvoiceDate|UnitPrice|CustomerID|       Country|
+---------+---------+--------------------+--------+--------------+---------+----------+--------------+
|   536365|   85123A|WHITE HANGING HEA...|       6|12/1/2010 8:26|     2.55|     17850|United Kingdom|
|   536365|    71053| WHITE METAL LANTERN|       6|12/1/2010 8:26|     3.39|     17850|United Kingdom|
|   536365|   84406B|CREAM CUPID HEART...|       8|12/1/2010 8:26|     2.75|     17850|United Kingdom|
|   536365|   84029G|KNITTED UNION FLA...|       6|12/1/2010 8:26|     3.39|     17850|United Kingdom|
|   536365|   84029E|RED WOOLLY HOTTIE...|       6|12/1/2010 8:26|     3.39|     17850|United Kingdom|
|   536365|    22752|SET 7 BABUSHKA NE...|       2|12/1/2010 8:26|     7.65|     17850|United Kingdom|
|   536365|    21730|GLASS STAR FROSTE...|       6|12/1/2010 8:26|     4.

In [4]:
df.count()

541909

# AGGREGATION FUNCTIONS

## COUNT

In [5]:
from pyspark.sql.functions import count
df.select(count('StockCode')).show()

+----------------+
|count(StockCode)|
+----------------+
|          541909|
+----------------+



## COUNT DISTINCT

In [6]:
from pyspark.sql.functions import *
df.select(countDistinct('StockCode')).show()

+-------------------------+
|count(DISTINCT StockCode)|
+-------------------------+
|                     4070|
+-------------------------+



## FIRST AND LAST

In [7]:
df.select(first('StockCode'),last('Stockcode')).show()

+----------------+---------------+
|first(StockCode)|last(Stockcode)|
+----------------+---------------+
|          85123A|          22138|
+----------------+---------------+



## MIN AND MAX

In [8]:
df.select(min('Quantity'),max('Quantity')).show()

+-------------+-------------+
|min(Quantity)|max(Quantity)|
+-------------+-------------+
|       -80995|        80995|
+-------------+-------------+



## SUM

In [9]:
df.select(sum('Quantity')).show()

+-------------+
|sum(Quantity)|
+-------------+
|      5176450|
+-------------+



In [10]:
df.select(sumDistinct('Quantity')).show();

/home/unik/spark-3.5.0-bin-hadoop3/python/pyspark/sql/functions.py:988: FutureWarning: Deprecated in 3.2, use sum_distinct instead.
  warnings.warn("Deprecated in 3.2, use sum_distinct instead.", FutureWarning)


+----------------------+
|sum(DISTINCT Quantity)|
+----------------------+
|                 29310|
+----------------------+



# AVG

In [11]:
df.select(avg('Quantity')).show()

+----------------+
|   avg(Quantity)|
+----------------+
|9.55224954743324|
+----------------+



In [12]:
df.select(
    count("Quantity").alias("total_transactions"),
    sum("Quantity").alias("total_purchases"),
    avg("Quantity").alias("avg_purchases"),
    expr("mean(Quantity)").alias("mean_purchases"))\
  .selectExpr(
    "total_purchases/total_transactions",
    "avg_purchases",
    "mean_purchases").show()

+--------------------------------------+----------------+----------------+
|(total_purchases / total_transactions)|   avg_purchases|  mean_purchases|
+--------------------------------------+----------------+----------------+
|                      9.55224954743324|9.55224954743324|9.55224954743324|
+--------------------------------------+----------------+----------------+



## VARIANCE AND STANDARD DEVIATION
    -The variance is the average of the squared differences from the mean, and the standard deviation is the square root of the variance

In [13]:
df.select(var_pop('Quantity'),var_samp('Quantity'),stddev_pop('Quantity'),stddev_samp('Quantity')).show()

+------------------+------------------+--------------------+---------------------+
| var_pop(Quantity)|var_samp(Quantity)|stddev_pop(Quantity)|stddev_samp(Quantity)|
+------------------+------------------+--------------------+---------------------+
|47559.303646609056|47559.391409298754|  218.08095663447796|   218.08115785023418|
+------------------+------------------+--------------------+---------------------+



## SKEWNESS AND KURTOSIS
    -Skewness and kurtosis are both measurements of extreme points in your data. 
    -Skewness measures the asymmetry of the values in your data around the mean, whereas kurtosis is a measure of the tail of data.

In [14]:
    df.select(skewness('Quantity'),kurtosis('Quantity')).show()

+-------------------+------------------+
| skewness(Quantity)|kurtosis(Quantity)|
+-------------------+------------------+
|-0.2640755761052562|119768.05495536952|
+-------------------+------------------+



## COVARIANCE AND CORRELATION
    -Correlation measures the Pearson correlation coefficient, which is scaled between –1 and +1. The covariance is scaled according to  the inputs in the data.

In [15]:
df.select(corr("InvoiceNo","Quantity"),covar_samp("InvoiceNo","Quantity"),covar_pop("InvoiceNo","Quantity")).show()

[Stage 43:==================================>                       (3 + 2) / 5]

+-------------------------+-------------------------------+------------------------------+
|corr(InvoiceNo, Quantity)|covar_samp(InvoiceNo, Quantity)|covar_pop(InvoiceNo, Quantity)|
+-------------------------+-------------------------------+------------------------------+
|     4.912186085635685E-4|             1052.7280543902734|            1052.7260778741693|
+-------------------------+-------------------------------+------------------------------+



# GROUPING
    -We do this grouping in two phases. First we specify the column(s) on which we would like to group, and then we specify the aggregation(s).

In [16]:
df.groupBy('InvoiceNo','CustomerId').count().show()

+---------+----------+-----+
|InvoiceNo|CustomerId|count|
+---------+----------+-----+
|   536846|     14573|   76|
|   537026|     12395|   12|
|   537883|     14437|    5|
|   538068|     17978|   12|
|   538279|     14952|    7|
|   538800|     16458|   10|
|   538942|     17346|   12|
|  C539947|     13854|    1|
|   540096|     13253|   16|
|   540530|     14755|   27|
|   541225|     14099|   19|
|   541978|     13551|    4|
|   542093|     17677|   16|
|   536596|      NULL|    6|
|   537252|      NULL|    1|
|   538041|      NULL|    1|
|   537159|     14527|   28|
|   537213|     12748|    6|
|   538191|     15061|   16|
|  C539301|     13496|    1|
+---------+----------+-----+
only showing top 20 rows



## GROUPING WITH MAPS

In [17]:
df.groupBy('InvoiceNo').agg(
    expr('avg(Quantity)'),\
    expr('stddev_pop(Quantity)')
).show()

+---------+------------------+--------------------+
|InvoiceNo|     avg(Quantity)|stddev_pop(Quantity)|
+---------+------------------+--------------------+
|   536596|               1.5|  1.1180339887498947|
|   536938|33.142857142857146|  20.698023172885524|
|   537252|              31.0|                 0.0|
|   537691|              8.15|   5.597097462078001|
|   538041|              30.0|                 0.0|
|   538184|12.076923076923077|   8.142590198943392|
|   538517|3.0377358490566038|  2.3946659604837897|
|   538879|21.157894736842106|  11.811070444356483|
|   539275|              26.0|  12.806248474865697|
|   539630|20.333333333333332|  10.225241100118645|
|   540499|              3.75|  2.6653642652865788|
|   540540|2.1363636363636362|  1.0572457590557278|
|  C540850|              -1.0|                 0.0|
|   540976|10.520833333333334|   6.496760677872902|
|   541432|             12.25|  10.825317547305483|
|   541518| 23.10891089108911|  20.550782784878713|
|   541783|1

# WINDOW FUNCTIONS

In [19]:
df.show()

+---------+---------+--------------------+--------+--------------+---------+----------+--------------+
|InvoiceNo|StockCode|         Description|Quantity|   InvoiceDate|UnitPrice|CustomerID|       Country|
+---------+---------+--------------------+--------+--------------+---------+----------+--------------+
|   536365|   85123A|WHITE HANGING HEA...|       6|12/1/2010 8:26|     2.55|     17850|United Kingdom|
|   536365|    71053| WHITE METAL LANTERN|       6|12/1/2010 8:26|     3.39|     17850|United Kingdom|
|   536365|   84406B|CREAM CUPID HEART...|       8|12/1/2010 8:26|     2.75|     17850|United Kingdom|
|   536365|   84029G|KNITTED UNION FLA...|       6|12/1/2010 8:26|     3.39|     17850|United Kingdom|
|   536365|   84029E|RED WOOLLY HOTTIE...|       6|12/1/2010 8:26|     3.39|     17850|United Kingdom|
|   536365|    22752|SET 7 BABUSHKA NE...|       2|12/1/2010 8:26|     7.65|     17850|United Kingdom|
|   536365|    21730|GLASS STAR FROSTE...|       6|12/1/2010 8:26|     4.

In [23]:
from pyspark.sql.window import *
from pyspark.sql.functions import *
part=Window.partitionBy('Country').orderBy('UnitPrice')
df.withColumn('Row Number',row_number().over(part)).show()        

[Stage 53:=======================>                                  (2 + 3) / 5]

+---------+---------+--------------------+--------+----------------+---------+----------+---------+----------+
|InvoiceNo|StockCode|         Description|Quantity|     InvoiceDate|UnitPrice|CustomerID|  Country|Row Number|
+---------+---------+--------------------+--------+----------------+---------+----------+---------+----------+
|   554037|    22619|SET OF 6 SOLDIER ...|      80| 5/20/2011 14:13|      0.0|     12415|Australia|         1|
|   574138|    23234|BISCUIT TIN VINTA...|     216| 11/3/2011 11:26|      0.0|     12415|Australia|         2|
|   574469|    22385|JUMBO BAG SPACEBO...|      12| 11/4/2011 11:55|      0.0|     12431|Australia|         3|
|   553546|    21084|SET/6 COLLAGE PAP...|     240| 5/17/2011 15:42|     0.19|     12415|Australia|         4|
|   578459|    22338|STAR DECORATION P...|      96|11/24/2011 12:30|     0.19|     12388|Australia|         5|
|   545475|    22615|PACK OF 12 CIRCUS...|     432|  3/3/2011 10:59|     0.25|     12415|Australia|         6|
|

In [29]:
df.withColumn('Rank',rank().over(part)).show()

+---------+---------+--------------------+--------+----------------+---------+----------+---------+----+
|InvoiceNo|StockCode|         Description|Quantity|     InvoiceDate|UnitPrice|CustomerID|  Country|Rank|
+---------+---------+--------------------+--------+----------------+---------+----------+---------+----+
|   554037|    22619|SET OF 6 SOLDIER ...|      80| 5/20/2011 14:13|      0.0|     12415|Australia|   1|
|   574138|    23234|BISCUIT TIN VINTA...|     216| 11/3/2011 11:26|      0.0|     12415|Australia|   1|
|   574469|    22385|JUMBO BAG SPACEBO...|      12| 11/4/2011 11:55|      0.0|     12431|Australia|   1|
|   553546|    21084|SET/6 COLLAGE PAP...|     240| 5/17/2011 15:42|     0.19|     12415|Australia|   4|
|   578459|    22338|STAR DECORATION P...|      96|11/24/2011 12:30|     0.19|     12388|Australia|   4|
|   545475|    22615|PACK OF 12 CIRCUS...|     432|  3/3/2011 10:59|     0.25|     12415|Australia|   6|
|   545475|    21984|PACK OF 12 PINK P...|     432|  3/

In [32]:
df_dense=df.withColumn('Dense Rank',dense_rank().over(part))
df_dense.show()
df_dense.explain()

+---------+---------+--------------------+--------+----------------+---------+----------+---------+----------+
|InvoiceNo|StockCode|         Description|Quantity|     InvoiceDate|UnitPrice|CustomerID|  Country|Dense Rank|
+---------+---------+--------------------+--------+----------------+---------+----------+---------+----------+
|   554037|    22619|SET OF 6 SOLDIER ...|      80| 5/20/2011 14:13|      0.0|     12415|Australia|         1|
|   574138|    23234|BISCUIT TIN VINTA...|     216| 11/3/2011 11:26|      0.0|     12415|Australia|         1|
|   574469|    22385|JUMBO BAG SPACEBO...|      12| 11/4/2011 11:55|      0.0|     12431|Australia|         1|
|   553546|    21084|SET/6 COLLAGE PAP...|     240| 5/17/2011 15:42|     0.19|     12415|Australia|         2|
|   578459|    22338|STAR DECORATION P...|      96|11/24/2011 12:30|     0.19|     12388|Australia|         2|
|   545475|    22615|PACK OF 12 CIRCUS...|     432|  3/3/2011 10:59|     0.25|     12415|Australia|         3|
|